SetUp

notebook ini dijalankan di **google colab** (runtime → change runtime type → **t4 gpu**). cell pertama meng-clone repo dari github dan menyalin `dataset_seg.pkl` (hasil notebook 01, tidak ada di github karena 200 mb) dari google drive. simpan dulu `dataset_seg.pkl` di drive, lalu sesuaikan `PKL_DRIVE` di bawah. kalau dijalankan di laptop, cell ini tidak melakukan apa-apa.

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/dindawijayanto/parking-area-detection.git'
PKL_DRIVE = '/content/drive/MyDrive/parking-area-detection/dataset_seg.pkl'   # where dataset_seg.pkl is in your Drive

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/parking-area-detection'):
        subprocess.run(['git', 'clone', REPO_URL, '/content/parking-area-detection'], check=True)
    os.chdir('/content/parking-area-detection/notebooks')
    if not os.path.exists('../data/processed/dataset_seg.pkl'):
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs('../data/processed', exist_ok=True)
        shutil.copy(PKL_DRIVE, '../data/processed/dataset_seg.pkl')
print('working folder:', os.getcwd())

In [ ]:
# pip install opencv-python matplotlib numpy pandas scikit-image scikit-learn xgboost joblib   (all already on Colab)
import pickle
import time
import warnings
from pathlib import Path

import cv2 as cv
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier

%config InlineBackend.figure_formats = ['jpeg']
warnings.filterwarnings('ignore', message='.*mismatched devices.*')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src' / 'ml'))
import parking as pk   # pixel features, post-processing, bird's-eye geometry and free-space counter (shared with predict.py)

SEG_PKL = ROOT / 'data' / 'processed' / 'dataset_seg.pkl'
MODEL_DIR = ROOT / 'models'
OUT_DIR = ROOT / 'results' / 'progress5'

FLOOR, MOTOR, IGNORE = pk.FLOOR, pk.MOTOR, pk.IGNORE
GPU = shutil.which('nvidia-smi') is not None
PER_FRAME = 12000    # labeled pixels sampled from every (augmented) train frame
N_AUG = 2            # augmented copies of every train frame, on top of the original
RF_SAMPLES = 400000  # the random forest runs on the CPU (2 cores on Colab), so it gets a subsample
SEED = 0
rng = np.random.default_rng(SEED)

print('project folder:', ROOT)
print('GPU for XGBoost:', GPU, '| xgboost', xgb.__version__)

Step 1: Load Dataset

In [ ]:
with open(SEG_PKL, 'rb') as f:
    data = pickle.load(f)

X, Y, meta, geo = data['X'], data['Y'], data['meta'], data['geo']
train_idx = np.where(meta['split'] == 'train')[0]
val_idx = np.where(meta['split'] == 'val')[0]
test_idx = np.where(meta['split'] == 'test')[0]

print('X:', X.shape, X.dtype, '| Y:', Y.shape, Y.dtype)
print('train', len(train_idx), '| val', len(val_idx), '| test', len(test_idx))
pd.crosstab(meta['weather'], meta['split'], margins=True)

notebook ini memakai `dataset_seg.pkl` dari notebook 01: frame 960 × 540 bgr (`X`), mask label (`Y`, 0 lantai, 1 motor, 255 ignore), meta dan `geo` per video. split nya tetap **per hari**, sama seperti baseline di notebook 02, jadi hasil nya bisa dibandingkan langsung.

catatan penting: **train tidak punya frame dusk**, padahal test punya satu video dusk (20261104_evening_keiza). baseline naive bayes di notebook 02 menghitung 0 ruang kosong di video itu.

Step 2: Pixel Features

In [ ]:
names = pk.feature_names()
i = train_idx[0]
t0 = time.time()
F = pk.pixel_features(X[i])
print(f'{F.shape[2]} features per pixel, {time.time() - t0:.2f} s per frame')

show = ['V clahe', 'a', 'b', 'H cos*S', 'Vc local std s2', 'Vc gradient s1', 'Vc LoG s4', 'b blur s8']
plt.figure(figsize=(20, 9))
plt.subplot(3, 3, 1)
plt.imshow(X[i][:, :, ::-1])
plt.title(meta['frame'][i])
plt.axis('off')
for k, n in enumerate(show):
    plt.subplot(3, 3, k + 2)
    plt.imshow(F[:, :, names.index(n)], cmap='magma')
    plt.title(n)
    plt.axis('off')
plt.tight_layout()
plt.show()

model ml klasik tidak bisa belajar fitur sendiri dari gambar seperti cnn, jadi setiap piksel dideskripsikan dengan **fitur buatan tangan** (hand-crafted), yaitu hasil pcd dari notebook 02. cara ini sama dengan ide *trainable weka segmentation* / *ilastik*:

- **warna piksel**: b, g, r, hsv (hue dipakai sebagai cos h · s dan sin h · s, supaya merah di h ≈ 0 dan h ≈ 179 berdekatan dan hue diabaikan kalau saturasi rendah), lab, dan **chromaticity** r / (r+g+b), g / (r+g+b) yang tidak terpengaruh kecerahan.
- **v setelah clahe** (notebook 02 step 6), supaya kecerahan di sore / hujan lebih mirip siang.
- **filter bank multi-skala** (σ = 1, 2, 4, 8) pada v clahe: gaussian blur, besar gradien (sobel), laplacian of gaussian (tepi dan bintik), local std (tekstur), dan blur a, b, s (warna lingkungan sekitar piksel).

dengan filter di beberapa skala, satu piksel "melihat" tetangga nya. piksel lantai di tengah paving punya tekstur hexagon yang teratur, piksel motor dikelilingi tepi tajam dan pantulan.

Step 3: Training Pixels with Lighting Augmentation

In [ ]:
def photometric(img):
    # random lighting: darker (dusk, rain), warmer / colder, less or more colour, a little noise and blur
    x = img.astype(np.float32) / 255
    if rng.random() < 0.6:
        x = x ** rng.uniform(1.0, 2.2) * rng.uniform(0.4, 1.0)          # dusk-like: gamma + exposure down
    else:
        x = x * rng.uniform(0.8, 1.2)
    x = (x - x.mean()) * rng.uniform(0.7, 1.3) + x.mean()               # contrast
    x = x * rng.uniform(0.9, 1.1, size=3)                               # colour temperature (B, G, R gains)
    gray = x.mean(axis=2, keepdims=True)
    x = gray + (x - gray) * rng.uniform(0.6, 1.4)                       # saturation
    x = x + rng.normal(0, rng.uniform(0, 0.03), x.shape)                # sensor noise
    x = np.uint8(np.clip(x, 0, 1) * 255)
    if rng.random() < 0.2:
        x = cv.GaussianBlur(x, (5, 5), 0)                               # rain drops / soft focus
    return x


def sample_pixels(img, mask, n):
    valid = np.flatnonzero(mask != IGNORE)
    pick = rng.choice(valid, min(n, len(valid)), replace=False)
    F = pk.pixel_features(img).reshape(-1, len(names))
    return F[pick], (mask.ravel()[pick] == FLOOR).astype(np.uint8)       # target: 1 = floor, 0 = motorcycle


t0 = time.time()
Xs, ys = [], []
for i in train_idx:
    for k in range(N_AUG + 1):
        img = X[i] if k == 0 else photometric(X[i])
        f, y = sample_pixels(img, Y[i], PER_FRAME)
        Xs.append(f)
        ys.append(y)
X_train, y_train = np.concatenate(Xs), np.concatenate(ys)

Xs, ys = [], []
for i in val_idx:
    f, y = sample_pixels(X[i], Y[i], PER_FRAME)
    Xs.append(f)
    ys.append(y)
X_val, y_val = np.concatenate(Xs), np.concatenate(ys)
del Xs, ys

print(f'train pixels: {len(y_train):,} ({100 * y_train.mean():.1f}% floor) | val pixels: {len(y_val):,}'
      f' | {X_train.nbytes / 1e6:.0f} MB | {time.time() - t0:.0f} s')

In [ ]:
i = train_idx[0]
plt.figure(figsize=(20, 6))
plt.subplot(2, 4, 1)
plt.imshow(X[i][:, :, ::-1])
plt.title('original')
plt.axis('off')
for k in range(7):
    plt.subplot(2, 4, k + 2)
    plt.imshow(photometric(X[i])[:, :, ::-1])
    plt.title(f'augmented {k + 1}')
    plt.axis('off')
plt.tight_layout()
plt.show()

tidak semua piksel dipakai untuk training: dari setiap frame diambil acak 12.000 piksel berlabel (lantai atau motor). piksel yang bersebelahan hampir sama, jadi sampel ini sudah mewakili frame nya, dan training jadi jauh lebih cepat.

setiap frame train juga dibuat **2 salinan dengan cahaya acak** (augmentasi): 60% nya digelapkan dengan gamma dan exposure seperti sore / hujan, ditambah perubahan kontras, suhu warna, saturasi, noise dan blur. tujuan nya supaya model pernah melihat lantai dan motor dalam kondisi gelap, walaupun tidak ada video dusk di train. piksel val diambil dari frame asli tanpa augmentasi.

Step 4: Train Random Forest and XGBoost

In [ ]:
pick = rng.choice(len(y_train), min(RF_SAMPLES, len(y_train)), replace=False)
rf = RandomForestClassifier(n_estimators=100, max_depth=18, min_samples_leaf=10, max_features='sqrt',
                            class_weight='balanced', n_jobs=-1, random_state=SEED)
t0 = time.time()
rf.fit(X_train[pick], y_train[pick])
print(f'random forest: {time.time() - t0:.0f} s on {len(pick):,} pixels, '
      f'val pixel accuracy {rf.score(X_val, y_val):.3f}')

In [ ]:
xgbc = xgb.XGBClassifier(n_estimators=1500, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.8,
                         min_child_weight=5, reg_lambda=1.0, tree_method='hist',
                         device='cuda' if GPU else 'cpu', eval_metric='logloss', early_stopping_rounds=50,
                         scale_pos_weight=float((y_train == 0).sum() / max(1, (y_train == 1).sum())),
                         random_state=SEED)
t0 = time.time()
xgbc.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
print(f'xgboost: {time.time() - t0:.0f} s on {len(y_train):,} pixels, best iteration {xgbc.best_iteration}, '
      f'val pixel accuracy {(xgbc.predict(X_val) == y_val).mean():.3f}')

models = {'Random Forest': rf, 'XGBoost': xgbc}

In [ ]:
plt.figure(figsize=(20, 6))
for k, (name, m) in enumerate(models.items()):
    imp = pd.Series(m.feature_importances_, index=names).sort_values()[-15:]
    plt.subplot(1, 2, k + 1)
    plt.barh(imp.index, imp.values)
    plt.title(f'{name}: 15 most important features')
plt.tight_layout()
plt.show()

dua model ml klasik berbasis **decision tree** dibandingkan:

- **random forest** (bagging): 100 pohon keputusan, masing-masing dilatih pada sampel acak piksel dan subset acak fitur, lalu suara nya dirata-rata. tahan terhadap overfitting dan hampir tanpa tuning. di colab random forest scikit-learn hanya jalan di cpu (2 core), jadi dilatih dengan 400.000 piksel.
- **xgboost** (gradient boosting): pohon dibuat satu per satu, setiap pohon baru memperbaiki kesalahan pohon sebelum nya. dilatih di **gpu t4** (`device='cuda'`) dengan semua piksel train. **early stopping** pada piksel val menghentikan training saat val tidak membaik lagi selama 50 pohon.

lantai hanya ±28% piksel, jadi kelas nya diseimbangkan (`class_weight='balanced'` dan `scale_pos_weight`) supaya model tidak cuma menebak "motor". grafik **feature importance** menunjukkan fitur pcd mana yang paling dipakai model.

Step 5: Threshold and Post-Processing on the Validation Days

In [ ]:
def evaluate(masks, idx):
    # IoU per class summed over all frames, plus pixel accuracy (same as notebooks/02)
    inter, union, correct, total = np.zeros(2), np.zeros(2), 0, 0
    for p, i in zip(masks, idx):
        valid = Y[i] != IGNORE
        for cls in [FLOOR, MOTOR]:
            inter[cls] += np.sum((p == cls) & (Y[i] == cls) & valid)
            union[cls] += np.sum(((p == cls) | (Y[i] == cls)) & valid)
        correct += np.sum((p == Y[i]) & valid)
        total += np.sum(valid)
    iou = inter / np.maximum(union, 1)
    return {'IoU floor': iou[FLOOR], 'IoU motor': iou[MOTOR], 'mIoU': iou.mean(), 'pixel acc': correct / total}


def roi_of(i):
    roi = geo[meta['video_id'][i]]['roi_mask'].copy()
    roi[X[i].max(axis=2) == 0] = 0      # black border left by the alignment warp (same as predict.py)
    return roi


SP = {}   # SLIC superpixels per frame, computed once and shared by both models


def probabilities(model, idx):
    # P(floor) per pixel, raw and averaged per SLIC superpixel
    raw, smooth = [], []
    for i in idx:
        if i not in SP:
            SP[i] = pk.superpixels(X[i], roi_of(i))
        p = pk.floor_probability(model, X[i], roi_of(i))
        raw.append(p)
        smooth.append(pk.superpixel_smooth(p, SP[i]))
    return {'raw': raw, 'superpixels': smooth}


POST = {'none': ('raw', False), 'morphology': ('raw', True),
        'superpixels': ('superpixels', False), 'superpixels + morphology': ('superpixels', True)}
thresholds = np.round(np.arange(0.2, 0.81, 0.05), 2)

val_probs, choice, val_rows = {}, {}, []
for name, m in models.items():
    t0 = time.time()
    val_probs[name] = probabilities(m, val_idx)
    best = (-1, None, None)
    for post, (kind, morph) in POST.items():
        scores = [evaluate([pk.to_mask(p, roi_of(i), t, morph) for p, i in zip(val_probs[name][kind], val_idx)],
                           val_idx)['mIoU'] for t in thresholds]
        k = int(np.argmax(scores))
        val_rows.append([name, post, thresholds[k], round(scores[k], 3)])
        if scores[k] > best[0]:
            best = (scores[k], post, float(thresholds[k]))
    choice[name] = {'post': best[1], 'threshold': best[2]}
    print(f"{name}: best on val = {best[1]}, threshold {best[2]:.2f}, val mIoU {best[0]:.3f}  ({time.time() - t0:.0f} s)")

pd.DataFrame(val_rows, columns=['model', 'post-processing', 'best threshold', 'val mIoU'])

model mengeluarkan **probabilitas lantai** per piksel. dua hal dipilih di hari **val** (bukan test):

1. **threshold**: piksel dianggap lantai kalau P(lantai) > threshold.
2. **post-processing** (pcd setelah klasifikasi):
   - **superpixels**: probabilitas dirata-rata di setiap superpixel slic (ukuran 12 px, sama seperti alat labeling). karena batas superpixel mengikuti tepi benda, prediksi yang berbintik jadi rapi dan batas motor lebih tajam.
   - **morphology**: opening lalu closing dengan elips 7 × 7, sama seperti notebook 02.

kombinasi dengan val miou tertinggi yang dipakai di test.

Step 6: Test Results

In [ ]:
# baseline from notebooks/02 (step 8 and 9), same test days
results = [['Naive Bayes HS + texture + morphology (02)', 'morphology', 0.537, 0.757, 0.647, 0.811, 12.81],
           ['Naive Bayes HS + texture (02)', 'none', 0.480, 0.708, 0.594, 0.770, 10.12]]

test_masks = {}
free_table = meta.loc[test_idx, ['frame', 'video_id', 'weather', 'free spaces']].rename(
    columns={'free spaces': 'label'}).reset_index(drop=True)
free_table['label: (ROI - motor) / 1 motor'] = [round(pk.free_spaces(Y[i], geo[meta['video_id'][i]])['floor_area'], 1) for i in test_idx]
for name, m in models.items():
    kind, morph = POST[choice[name]['post']]
    probs = probabilities(m, test_idx)[kind]
    test_masks[name] = [pk.to_mask(p, roi_of(i), choice[name]['threshold'], morph) for p, i in zip(probs, test_idx)]
    r = evaluate(test_masks[name], test_idx)
    free_table[name] = [pk.count_free_spaces(mk, geo[meta['video_id'][i]]) for mk, i in zip(test_masks[name], test_idx)]
    free_table[f'{name}: (ROI - motor) / 1 motor'] = [round(pk.free_spaces(mk, geo[meta['video_id'][i]])['floor_area'], 1)
                                                     for mk, i in zip(test_masks[name], test_idx)]
    mae = float(np.mean(np.abs(free_table[name] - free_table['label'])))
    results.append([name, choice[name]['post']] + [round(r[k], 3) for k in ['IoU floor', 'IoU motor', 'mIoU', 'pixel acc']]
                   + [round(mae, 2)])

results = pd.DataFrame(results, columns=['model', 'post-processing', 'IoU floor', 'IoU motor', 'mIoU', 'pixel acc',
                                         'MAE free spaces'])
results

In [ ]:
per_weather = free_table.copy()
for name in models:
    per_weather[f'abs error {name}'] = np.abs(per_weather[name] - per_weather['label'])
per_weather.groupby('weather')[['label'] + list(models) + [f'abs error {n}' for n in models]].mean().round(1)

In [ ]:
plt.figure(figsize=(18, 4))
x = np.arange(len(free_table))
plt.plot(x, free_table['label'], 'ko-', label='label (mask by hand)')
for name in models:
    plt.plot(x, free_table[name], 'o--', label=name)
top = max(free_table[['label'] + list(models)].max()) + 3
for v in free_table['video_id'].unique():
    first = int(np.where(free_table['video_id'] == v)[0][0])
    plt.axvline(first - 0.5, color='gray', lw=0.5)
    plt.text(first, top, v, fontsize=8)
plt.xlabel('test frame')
plt.ylabel('free spaces')
plt.title('free spaces per test frame: label vs prediction')
plt.legend(loc='lower right')
plt.show()
free_table

semua model dinilai di hari **test** yang sama dengan baseline naive bayes dari notebook 02, dengan dua ukuran:

- **iou / pixel accuracy**: seberapa tepat mask lantai dan motor per piksel.
- **(ROI − motor) / 1 motor**: rumus paling sederhana, luas lantai kosong di bird's-eye view (luas roi dikurangi luas yang tertutup motor) dibagi luas satu motor, tanpa opening. kolom **free spaces** adalah versi setelah opening, yang hanya menghitung lantai yang benar-benar muat untuk motor (celah sempit di antara motor tidak dihitung).
- **mae free spaces**: rata-rata selisih jumlah ruang kosong per frame terhadap jumlah dari mask label. ini ukuran yang paling penting, karena yang akhirnya ditampilkan ke pengguna adalah jumlah tempat kosong.

tabel per cuaca menunjukkan apakah model masih benar saat **dusk**, kondisi yang tidak ada di train.

Step 7: Predicted Free Parking Areas

In [ ]:
BEST = min(models, key=lambda n: float(np.mean(np.abs(free_table[n] - free_table['label']))))
print('best model on the test days (lowest MAE):', BEST)

OUT_DIR.mkdir(parents=True, exist_ok=True)
show = [int(test_idx[np.where(free_table['video_id'] == v)[0][0]]) for v in free_table['video_id'].unique()]

plt.figure(figsize=(20, 5.6 * len(show)))
for r, i in enumerate(show):
    k = list(test_idx).index(i)
    g = geo[meta['video_id'][i]]
    label_vis = pk.draw_free_spaces(X[i], Y[i], g)
    pred_vis = pk.draw_free_spaces(X[i], test_masks[BEST][k], g)
    cv.imwrite(str(OUT_DIR / f"free_{meta['frame'][i]}"), np.hstack([label_vis, pred_vis]))
    plt.subplot(len(show), 2, 2 * r + 1)
    plt.imshow(label_vis[:, :, ::-1])
    plt.title(f"{meta['frame'][i]}: from the label mask")
    plt.axis('off')
    plt.subplot(len(show), 2, 2 * r + 2)
    plt.imshow(pred_vis[:, :, ::-1])
    plt.title(f'predicted by {BEST}')
    plt.axis('off')
plt.tight_layout()
plt.show()
print('saved in', OUT_DIR)

ini output akhir nya: **daerah parkir kosong** di tampilan kamera.

- **hijau**: lantai di dalam roi yang masih muat untuk motor (hasil opening dengan tapak satu motor di bird's-eye view, lalu dikembalikan ke tampilan kamera dengan homography terbalik).
- **merah**: piksel yang diprediksi motor.
- **angka** di setiap area hijau: berapa motor yang muat di area itu (luas area ÷ luas satu motor), dan total nya di pojok kiri atas.

jadi luas lantai kosong = luas roi − luas yang tertutup motor, dihitung di bird's-eye view supaya setiap piksel mewakili luas lantai yang sama, lalu dikonversi ke jumlah motor.

Step 8: Save the Models

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
for name, m in models.items():
    if name == 'XGBoost':
        m.set_params(device='cpu')   # so the saved model also runs on a laptop without GPU
    kind, morph = POST[choice[name]['post']]
    row = results[results['model'] == name].iloc[0]
    bundle = {'name': name, 'model': m, 'threshold': choice[name]['threshold'],
              'superpixels': kind == 'superpixels', 'morphology': morph, 'feature_names': names,
              'test mIoU': float(row['mIoU']), 'test MAE free spaces': float(row['MAE free spaces'])}
    path = MODEL_DIR / (name.lower().replace(' ', '_') + '.joblib')
    joblib.dump(bundle, path, compress=3)
    print('saved', path, f'({path.stat().st_size / 1e6:.0f} MB)')
results.to_csv(OUT_DIR / 'results_test.csv', index=False)
free_table.to_csv(OUT_DIR / 'free_spaces_test.csv', index=False)

In [ ]:
# Colab: download the models and the result pictures (they disappear when the runtime stops)
if 'google.colab' in sys.modules:
    shutil.make_archive('/content/progress5', 'zip', ROOT, 'results/progress5')
    shutil.make_archive('/content/models', 'zip', ROOT, 'models')
    from google.colab import files
    files.download('/content/progress5.zip')
    files.download('/content/models.zip')

model disimpan di `models/random_forest.joblib` dan `models/xgboost.joblib` (tidak di-commit, `*.joblib` ada di `.gitignore`), bersama threshold dan post-processing yang dipilih di val. di colab, file nya di-download sebagai zip. untuk memprediksi frame atau video baru dari posisi kamera yang sama:

```
python src/ml/predict.py 20261112_morning_keiza data/raw/frames/20261112_morning_keiza_t000.jpg
python src/ml/predict.py 20261112_morning_keiza data/raw/video_baru.mp4 --every 5 --model models/random_forest.joblib
```

hasil nya ada di `results/predictions/`: gambar `<nama>_free.jpg` seperti step 7 dan `free_spaces.csv` dengan jumlah ruang kosong per gambar.